# StockPulse - Week 2: Data Exploration

**Goal:** Understand what data is available from yfinance and define our scoring methodology

**Test Stocks:** AAPL, MSFT, GOOGL, AMZN, NVDA

## Setup

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import time

In [ ]:
# Enable caching to avoid rate limits
# Once data is fetched successfully, it will be cached locally
# This means we can re-run cells without hitting Yahoo Finance again!
import requests_cache

# Set up cache (expires after 1 day)
requests_cache.install_cache('yfinance_cache', expire_after=86400)
print("✅ Caching enabled! Data will be saved locally after first fetch.")

In [ ]:
# Test stocks
test_tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA']

## 1. Explore Data Structure

Let's fetch data for AAPL first and see what's available

In [ ]:
# Fetch AAPL data
ticker = yf.Ticker("AAPL")

# Basic info
info = ticker.info
print("Available info keys:")
print(list(info.keys())[:20])  # Show first 20 keys

In [ ]:
# Historical prices
history = ticker.history(period="1y")
print("\nPrice history (last 5 days):")
print(history.tail())

In [ ]:
# Key metrics we care about
print("\n=== Key Financial Metrics ===")
print(f"Company: {info.get('longName', 'N/A')}")
print(f"Sector: {info.get('sector', 'N/A')}")
print(f"Market Cap: ${info.get('marketCap', 0) / 1e9:.2f}B")
print(f"\nValuation:")
print(f"  P/E Ratio: {info.get('trailingPE', 'N/A')}")
print(f"  P/B Ratio: {info.get('priceToBook', 'N/A')}")
print(f"  P/S Ratio: {info.get('priceToSalesTrailing12Months', 'N/A')}")
print(f"\nProfitability:")
print(f"  ROE: {info.get('returnOnEquity', 'N/A')}")
print(f"  Profit Margin: {info.get('profitMargins', 'N/A')}")
print(f"\nFinancial Health:")
print(f"  Debt/Equity: {info.get('debtToEquity', 'N/A')}")
print(f"  Current Ratio: {info.get('currentRatio', 'N/A')}")

## 2. Fetch Data for All Test Stocks

Run this cell to get data for all 5 test stocks

In [ ]:
# Function to fetch stock data
def fetch_stock_data(ticker_symbol):
    """Fetch key metrics for a stock"""
    try:
        ticker = yf.Ticker(ticker_symbol)
        info = ticker.info
        
        # Extract key metrics
        data = {
            'ticker': ticker_symbol,
            'name': info.get('longName', 'N/A'),
            'sector': info.get('sector', 'N/A'),
            'market_cap': info.get('marketCap', None),
            'pe_ratio': info.get('trailingPE', None),
            'pb_ratio': info.get('priceToBook', None),
            'ps_ratio': info.get('priceToSalesTrailing12Months', None),
            'roe': info.get('returnOnEquity', None),
            'profit_margin': info.get('profitMargins', None),
            'debt_to_equity': info.get('debtToEquity', None),
            'current_ratio': info.get('currentRatio', None),
            'revenue_growth': info.get('revenueGrowth', None),
        }
        
        # Get price history for momentum
        history = ticker.history(period="6mo")
        if not history.empty:
            current_price = history['Close'].iloc[-1]
            price_3m_ago = history['Close'].iloc[-63] if len(history) >= 63 else history['Close'].iloc[0]
            data['return_3m'] = (current_price - price_3m_ago) / price_3m_ago
        
        return data
    
    except Exception as e:
        print(f"Error fetching {ticker_symbol}: {e}")
        return None

# Fetch data for all test stocks
stock_data = []

for ticker in test_tickers:
    print(f"Fetching {ticker}...")
    data = fetch_stock_data(ticker)
    if data:
        stock_data.append(data)
    time.sleep(1)  # Be nice to the API

# Create DataFrame
df = pd.DataFrame(stock_data)
print("\n=== Stock Data Summary ===")
print(df)

## 3. Data Quality Check

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nData completeness: {(1 - df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100:.1f}%")

## 4. Initial Scoring Exploration

Let's try some basic scoring to see if it makes sense

In [ ]:
# Simple value score (lower P/E is better, normalize 0-100)
def calculate_value_score_simple(df):
    """Simple value score based on P/E ratio"""
    if 'pe_ratio' in df.columns and df['pe_ratio'].notna().any():
        pe_scores = 100 - (df['pe_ratio'] - df['pe_ratio'].min()) / (df['pe_ratio'].max() - df['pe_ratio'].min()) * 100
        return pe_scores
    return pd.Series([50] * len(df))

df['value_score'] = calculate_value_score_simple(df)
print("Value Scores:")
print(df[['ticker', 'name', 'pe_ratio', 'value_score']])

## 5. Next Steps & Notes

**Add your observations here as you work:**
- What fields are available?
- Any data quality issues?
- Which metrics seem most useful?
- Ideas for scoring methodology?

**TODO:**
1. Add more scoring functions (growth, profitability, momentum, quality)
2. Document findings in `docs/data_sources.md`
3. Create financial metrics cheat sheet in `docs/finance_101.md`
4. Write scoring methodology in `docs/scoring_methodology.md`